In [1]:

# Restart with optimized approach - load data and coefficients more efficiently
import numpy as np
import pickle
import time
from numba import jit
import warnings
warnings.filterwarnings('ignore')

# Load artifacts
print("Loading pre-computed artifacts...")
with open('f_canon_rand_primes_N1e7.pkl', 'rb') as f:
 prime_data = pickle.load(f)
primes = prime_data['primes']
a_p = prime_data['a_p']

with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_1e6 = pickle.load(f)
 
with open('omega_values_N1e7.pkl', 'rb') as f:
 omega_1e7 = pickle.load(f)

print(f"Loaded {len(primes)} primes")
print(f"Loaded Ω values for N=10^6 and N=10^7")


Loading pre-computed artifacts...


Loaded 664579 primes
Loaded Ω values for N=10^6 and N=10^7


In [2]:

# Generate coefficients - optimized version
def generate_multiplicative_coeffs(N, primes, a_p):
 """Generate completely multiplicative coefficients efficiently"""
 coeffs = np.ones(N, dtype=np.complex128)
 
 for i, p in enumerate(primes):
 if p > N:
 break
 a = a_p[i]
 # For each prime power
 pk = p
 while pk <= N:
 # Multiply all multiples of pk by a
 for m in range(pk, N + 1, pk):
 coeffs[m - 1] *= a
 pk *= p
 
 return coeffs

# Generate for both N values
print("Generating coefficients...")
t0 = time.time()
coeffs_1e6 = generate_multiplicative_coeffs(int(1e6), primes, a_p)
print(f"N=10^6: {time.time()-t0:.1f}s")

t0 = time.time()
coeffs_1e7 = generate_multiplicative_coeffs(int(1e7), primes, a_p)
print(f"N=10^7: {time.time()-t0:.1f}s")


Generating coefficients...


N=10^6: 0.7s


N=10^7: 7.7s


In [3]:

# Numba-accelerated Dirichlet sum
@jit(nopython=True)
def dirichlet_sum_numba(coeffs, t, N):
 """Compute D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it)"""
 result = 0.0 + 0.0j
 for n in range(1, N + 1):
 log_n = np.log(n)
 phase = t * log_n
 sqrt_n = np.sqrt(n)
 cos_phase = np.cos(phase)
 sin_phase = np.sin(phase)
 a_n = coeffs[n - 1]
 real_part = (a_n.real * cos_phase + a_n.imag * sin_phase) / sqrt_n
 imag_part = (a_n.imag * cos_phase - a_n.real * sin_phase) / sqrt_n
 result += real_part + 1j * imag_part
 return result

# Test
print("Testing Numba summation...")
D_test = dirichlet_sum_numba(coeffs_1e6, 1e6, int(1e6))
print(f"D(10^6; 10^6) = {D_test}, |D| = {np.abs(D_test):.4f}")


Testing Numba summation...


D(10^6; 10^6) = (0.03800477853308566-0.05214735558485198j), |D| = 0.0645


In [4]:

# Grid search at N=10^6 - save intermediate results to avoid timeout issues
N_1e6 = int(1e6)
t_grid_1e6 = np.linspace(N_1e6, 2*N_1e6, 1000)

print(f"Grid search at N=10^6: 1000 points from {t_grid_1e6[0]:.0f} to {t_grid_1e6[-1]:.0f}")
print(f"Spacing: {t_grid_1e6[1]-t_grid_1e6[0]:.1f}")

# Do search in batches
batch_size = 100
n_batches = len(t_grid_1e6) // batch_size
magnitudes_1e6 = np.zeros(len(t_grid_1e6))
values_1e6 = np.zeros(len(t_grid_1e6), dtype=np.complex128)

t_start = time.time()
for batch_idx in range(n_batches):
 i_start = batch_idx * batch_size
 i_end = min(i_start + batch_size, len(t_grid_1e6))
 
 for i in range(i_start, i_end):
 D = dirichlet_sum_numba(coeffs_1e6, t_grid_1e6[i], N_1e6)
 magnitudes_1e6[i] = np.abs(D)
 values_1e6[i] = D
 
 elapsed = time.time() - t_start
 progress = (i_end / len(t_grid_1e6)) * 100
 rate = i_end / elapsed
 remaining = (len(t_grid_1e6) - i_end) / rate
 
 print(f"Batch {batch_idx+1}/{n_batches}: {progress:.1f}% complete, "
 f"elapsed {elapsed:.0f}s, remaining ~{remaining:.0f}s")

print(f"\nCompleted in {time.time()-t_start:.1f}s")
print(f"Max |D| = {magnitudes_1e6.max():.4f} at t = {t_grid_1e6[np.argmax(magnitudes_1e6)]:.1f}")


Grid search at N=10^6: 1000 points from 1000000 to 2000000
Spacing: 1001.0


Batch 1/10: 10.0% complete, elapsed 3s, remaining ~29s


Batch 2/10: 20.0% complete, elapsed 6s, remaining ~26s


Batch 3/10: 30.0% complete, elapsed 10s, remaining ~22s


Batch 4/10: 40.0% complete, elapsed 13s, remaining ~19s


...

Batch 6/10: 60.0% complete, elapsed 19s, remaining ~13s


Batch 7/10: 70.0% complete, elapsed 23s, remaining ~10s


Batch 8/10: 80.0% complete, elapsed 26s, remaining ~6s


Batch 9/10: 90.0% complete, elapsed 29s, remaining ~3s


Batch 10/10: 100.0% complete, elapsed 33s, remaining ~0s

Completed in 32.6s
Max |D| = 42.4225 at t = 1543543.5


In [5]:

# Get top 50 peaks for N=10^6
top_50_idx_1e6 = np.argsort(magnitudes_1e6)[-50:][::-1]
peaks_1e6 = {
 't': t_grid_1e6[top_50_idx_1e6],
 'magnitude': magnitudes_1e6[top_50_idx_1e6],
 'D': values_1e6[top_50_idx_1e6]
}

print("Top 50 peaks at N=10^6:")
for i in range(10):
 print(f"{i+1:2d}. t={peaks_1e6['t'][i]:10.1f}, |D|={peaks_1e6['magnitude'][i]:7.4f}")
print(" ...")
for i in range(45, 50):
 print(f"{i+1:2d}. t={peaks_1e6['t'][i]:10.1f}, |D|={peaks_1e6['magnitude'][i]:7.4f}")


Top 50 peaks at N=10^6:
 1. t= 1543543.5, |D|=42.4225
 2. t= 1216216.2, |D|=28.6153
 3. t= 1221221.2, |D|=22.6286
 4. t= 1548548.5, |D|=21.1950
 5. t= 1905905.9, |D|=19.3493
 6. t= 1863863.9, |D|=19.3354
 7. t= 1585585.6, |D|=18.9140
 8. t= 1494494.5, |D|=17.6858
 9. t= 1382382.4, |D|=17.1246
10. t= 1910910.9, |D|=15.1060
 ...
46. t= 1744744.7, |D|= 7.5701
47. t= 1779779.8, |D|= 7.5200
48. t= 1431431.4, |D|= 7.5128
49. t= 1745745.7, |D|= 7.4175
50. t= 1627627.6, |D|= 7.4023


In [6]:

# Now the same for N=10^7
N_1e7 = int(1e7)
t_grid_1e7 = np.linspace(N_1e7, 2*N_1e7, 1000)

print(f"Grid search at N=10^7: 1000 points from {t_grid_1e7[0]:.0f} to {t_grid_1e7[-1]:.0f}")
print(f"Spacing: {t_grid_1e7[1]-t_grid_1e7[0]:.1f}")

batch_size = 100
n_batches = len(t_grid_1e7) // batch_size
magnitudes_1e7 = np.zeros(len(t_grid_1e7))
values_1e7 = np.zeros(len(t_grid_1e7), dtype=np.complex128)

t_start = time.time()
for batch_idx in range(n_batches):
 i_start = batch_idx * batch_size
 i_end = min(i_start + batch_size, len(t_grid_1e7))
 
 for i in range(i_start, i_end):
 D = dirichlet_sum_numba(coeffs_1e7, t_grid_1e7[i], N_1e7)
 magnitudes_1e7[i] = np.abs(D)
 values_1e7[i] = D
 
 elapsed = time.time() - t_start
 progress = (i_end / len(t_grid_1e7)) * 100
 rate = i_end / elapsed
 remaining = (len(t_grid_1e7) - i_end) / rate
 
 print(f"Batch {batch_idx+1}/{n_batches}: {progress:.1f}% complete, "
 f"elapsed {elapsed:.0f}s, remaining ~{remaining:.0f}s")

print(f"\nCompleted in {time.time()-t_start:.1f}s")
print(f"Max |D| = {magnitudes_1e7.max():.4f} at t = {t_grid_1e7[np.argmax(magnitudes_1e7)]:.1f}")


Grid search at N=10^7: 1000 points from 10000000 to 20000000
Spacing: 10010.0


Batch 1/10: 10.0% complete, elapsed 84s, remaining ~759s


Batch 2/10: 20.0% complete, elapsed 169s, remaining ~674s


Batch 3/10: 30.0% complete, elapsed 253s, remaining ~590s


Batch 4/10: 40.0% complete, elapsed 337s, remaining ~505s


...

Batch 6/10: 60.0% complete, elapsed 505s, remaining ~337s


Batch 7/10: 70.0% complete, elapsed 589s, remaining ~252s


Batch 8/10: 80.0% complete, elapsed 673s, remaining ~168s


Batch 9/10: 90.0% complete, elapsed 757s, remaining ~84s


Batch 10/10: 100.0% complete, elapsed 841s, remaining ~0s

Completed in 840.6s
Max |D| = 35.9521 at t = 15295295.3


In [7]:

# Get top 50 peaks for N=10^7
top_50_idx_1e7 = np.argsort(magnitudes_1e7)[-50:][::-1]
peaks_1e7 = {
 't': t_grid_1e7[top_50_idx_1e7],
 'magnitude': magnitudes_1e7[top_50_idx_1e7],
 'D': values_1e7[top_50_idx_1e7]
}

print("Top 50 peaks at N=10^7:")
for i in range(10):
 print(f"{i+1:2d}. t={peaks_1e7['t'][i]:11.1f}, |D|={peaks_1e7['magnitude'][i]:7.4f}")
print(" ...")
for i in range(45, 50):
 print(f"{i+1:2d}. t={peaks_1e7['t'][i]:11.1f}, |D|={peaks_1e7['magnitude'][i]:7.4f}")


Top 50 peaks at N=10^7:
 1. t= 15295295.3, |D|=35.9521
 2. t= 12612612.6, |D|=30.8924
 3. t= 17777777.8, |D|=25.5229
 4. t= 18388388.4, |D|=24.6042
 5. t= 11501501.5, |D|=23.2558
 6. t= 13063063.1, |D|=19.7863
 7. t= 11821821.8, |D|=16.7584
 8. t= 10650650.7, |D|=16.0891
 9. t= 12922922.9, |D|=15.1455
10. t= 17317317.3, |D|=14.7348
 ...
46. t= 11041041.0, |D|= 8.6214
47. t= 18148148.1, |D|= 8.6173
48. t= 18918918.9, |D|= 8.5781
49. t= 19979980.0, |D|= 8.5432
50. t= 15415415.4, |D|= 8.4890


In [8]:

# Compute omega-class decomposition and r metric for N=10^6
def compute_omega_decomposition_and_r(D_values, t_values, coeffs, omega_values, N):
 """
 Compute omega-class decomposition and inter-class energy ratio r.
 Returns: list of dicts with r, adjacent/non-adjacent contributions for each peak
 """
 results = []
 
 for idx in range(len(t_values)):
 t = t_values[idx]
 D_total = D_values[idx]
 
 # Compute S_k for each omega class
 omega_max = np.max(omega_values[:N])
 S_k = np.zeros(omega_max + 1, dtype=np.complex128)
 
 for n in range(1, N + 1):
 k = omega_values[n - 1] # 0-based indexing
 
 log_n = np.log(n)
 phase = t * log_n
 sqrt_n = np.sqrt(n)
 cos_phase = np.cos(phase)
 sin_phase = np.sin(phase)
 
 a_n = coeffs[n - 1]
 term_real = (a_n.real * cos_phase + a_n.imag * sin_phase) / sqrt_n
 term_imag = (a_n.imag * cos_phase - a_n.real * sin_phase) / sqrt_n
 
 S_k[k] += term_real + 1j * term_imag
 
 # Compute r metric: Σ_{j≠k} Re[S_j S̄_k] / Σ_k |S_k|²
 denominator = np.sum(np.abs(S_k)**2)
 
 numerator_total = 0.0
 numerator_adjacent = 0.0
 numerator_nonadjacent = 0.0
 
 for j in range(len(S_k)):
 for k in range(len(S_k)):
 if j != k:
 contrib = np.real(S_k[j] * np.conj(S_k[k]))
 numerator_total += contrib
 
 if abs(j - k) == 1:
 numerator_adjacent += contrib
 else:
 numerator_nonadjacent += contrib
 
 r_value = numerator_total / denominator if denominator > 0 else 0.0
 
 pct_adjacent = (numerator_adjacent / numerator_total * 100) if numerator_total != 0 else 0
 pct_nonadjacent = (numerator_nonadjacent / numerator_total * 100) if numerator_total != 0 else 0
 
 results.append({
 't': t,
 'magnitude': np.abs(D_total),
 'r': r_value,
 'pct_adjacent': pct_adjacent,
 'pct_nonadjacent': pct_nonadjacent
 })
 
 return results

print("Computing omega-class decomposition for N=10^6 (50 peaks)...")
t_start = time.time()
results_1e6 = compute_omega_decomposition_and_r(
 peaks_1e6['D'], peaks_1e6['t'], coeffs_1e6, omega_1e6, N_1e6
)
print(f"Completed in {time.time()-t_start:.1f}s")


TimeoutError: Code execution timed out after 51 seconds